# Breast Thermography 3D Reconstruction (DMR-IR) - Orthographic Geometric Pipeline

**Scope:** Pure geometric 2.5D reconstruction from five segmented thermal views (0 deg, +/-45 deg, +/-90 deg).  
**Goal:** Reconstruct a smooth 3D NURBS-like breast surface and extract asymmetry descriptors for downstream NNMF-ReliefF and Ensemble CNN analysis.  
**Framework:** NumPy + PyTorch + SciPy (edge-computing friendly).

---

## Protocol lock (Moura VISAPP 2024/2025 aligned)

1. Input is a 1-pixel skeletonized contour from semantic segmentation (256x256).
2. Phase 3 uses anatomical anchors (P1, P2, P3) with strict lateral clipping.
3. Phase 4 rotates around anatomical pivots (never camera center).
4. Phase 4 follows Algorithm 1 rigid registration: move local P2 to origin, rotate, then translate to frontal P1 or P3 target anchor.
5. Phase 5 surface interpolation uses B-spline fitting with degree 4 (`kx=4`, `ky=4`) and evaluation step `0.01` in parametric space.
6. Phase 6 outputs numeric geometry features, not only rendered figures.

This notebook includes reference functions that follow these constraints directly.

---

## Phase 1 - Data Loading and Normalization

This phase pairs each thermal TIFF with its binary mask, resizes both to $256 \times 256$, and converts them to tensors suitable for U-Net input.

The thermal image is min-max normalized to preserve temperature contrast while mapping values into a stable learning range:

$$
m_{ij} = \frac{P_{ij} - \min(P)}{\max(P) - \min(P)}
$$

This is equivalent to the paper's normalized intensity step before model training.

In [ ]:
%pip install torch torchvision tifffile opencv-python scikit-learn matplotlib pandas seaborn tensorflow keras cv2


In [ ]:
pip install opencv-python

### 1.1 ThermalDataset class

The dataset class performs three important jobs:

1. Finds mask files recursively and uses them as the reference sample list.
2. Locates each TIFF robustly using filename-prefix matching to avoid Unicode mismatches (for example degree-symbol variants).
3. Returns image-mask pairs as shape $(1, H, W)$ tensors so they are immediately compatible with the U-Net input/output format.

This design is intentionally defensive for Windows path and encoding edge cases.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, random_split
import tifffile as tiff
import cv2
import numpy as np
import os
import glob

# ── FIX 2: Reproducibility seed ──────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


class ThermalDataset(Dataset):
    def __init__(self, tiff_dir, mask_dir):
        # We find all masks first as they represent our ground truth batch
        self.mask_paths = glob.glob(os.path.join(mask_dir, "**", "*.png"), recursive=True)
        self.tiff_dir = tiff_dir
        self.mask_dir = mask_dir
        print(f"Dataset initialized with {len(self.mask_paths)} masks.")

    def __len__(self):
        return len(self.mask_paths)

    def __getitem__(self, idx):
        mask_path = self.mask_paths[idx]

        # 1. Get the patient folder and filename (e.g., Patient_106/benign/Right Lateral...)
        rel_path = os.path.relpath(mask_path, self.mask_dir)
        patient_subfolder = os.path.dirname(rel_path)
        mask_filename = os.path.basename(rel_path)

        # 2. Robust TIFF Search: Find the TIFF that matches the mask name
        # We search the folder directly to avoid encoding mismatches with the degree symbol
        tiff_search_folder = os.path.join(self.tiff_dir, patient_subfolder)
        base_name = os.path.splitext(mask_filename)[0]

        # We use a wildcard search to find the .tiff file matching the start of the mask name
        # This bypasses the specific encoding of the (90°) part
        search_pattern = os.path.join(tiff_search_folder, "*.tiff")
        potential_tiffs = glob.glob(search_pattern)

        tiff_path = None
        for p_tiff in potential_tiffs:
            if base_name[:10] in os.path.basename(p_tiff):  # Match first 10 chars (e.g., 'Right Late')
                tiff_path = p_tiff
                break

        if tiff_path is None or not os.path.exists(tiff_path):
            raise FileNotFoundError(f"Could not find matching TIFF for {mask_path}")

        # 3. Load and Normalize TIFF (Paper Eq 1) [cite: 113]
        raw_data = tiff.imread(tiff_path)
        t_min, t_max = np.min(raw_data), np.max(raw_data)

        # Prevent division by zero
        if t_max == t_min:
            normalized = np.zeros_like(raw_data, dtype=np.float32)
        else:
            normalized = (raw_data - t_min) / (t_max - t_min)

        img_256 = cv2.resize(normalized.astype(np.float32), (256, 256), interpolation=cv2.INTER_AREA)

        # 4. Load Mask
        # Use np.fromfile to bypass Windows encoding issues with the degree symbol (°)
        img_array = np.fromfile(mask_path, dtype=np.uint8)
        mask = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise ValueError(f"Could not read mask file (might be corrupted or empty): {mask_path}")
        mask_256 = cv2.resize(mask, (256, 256)).astype(np.float32) / 255.0

        # Return as (C, H, W) tensors
        return torch.from_numpy(img_256).unsqueeze(0), torch.from_numpy(mask_256).unsqueeze(0)


# Paths in Thinkpad environment (adjust as needed)
TIFF_BASE = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\organized_by_patient"
MASK_BASE = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\GroundTruth_Masks"

# Paths in Thinkpad environment (adjust as needed)
# TIFF_BASE = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\data\organized_by_patient"
# MASK_BASE = r"C:\Users\User\Documents\Skripsi Faiz_DL\DNP-3DDMR-IR\data\GroundTruth_Masks"

# ── FIX 1: Train / test split — 78 % / 22 % per paper §3.3 ──────────────────
full_dataset = ThermalDataset(TIFF_BASE, MASK_BASE)
n_total = len(full_dataset)
n_train = int(0.78 * n_total)
n_test  = n_total - n_train
train_set, test_set = random_split(
    full_dataset, [n_train, n_test],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"Split: {n_train} train / {n_test} test  (total={n_total})")

train_loader = DataLoader(train_set, batch_size=2, shuffle=True)   # Batch size 2 as per paper
test_loader  = DataLoader(test_set,  batch_size=2, shuffle=False)


### 1.2 Training-time augmentation

To reduce overfitting on a limited medical dataset, augmentation is applied only to training batches.

- Horizontal flip is applied to both image and mask so spatial correspondence remains correct.
- Brightness jitter is applied only to the thermal image.

This keeps anatomical geometry valid while improving generalization.

In [ ]:
# ── FIX 3: Data augmentation ─────────────────────────────────────────────────
def augment(img_tensor, mask_tensor):
    """Random horizontal flip + brightness jitter. Applied to both image and mask."""
    # Random horizontal flip — applied identically to image and mask
    if torch.rand(1).item() > 0.5:
        img_tensor  = TF.hflip(img_tensor)
        mask_tensor = TF.hflip(mask_tensor)

    # Random brightness jitter on image only (masks are binary)
    if torch.rand(1).item() > 0.5:
        factor     = 0.8 + torch.rand(1).item() * 0.4  # uniform in [0.8, 1.2]
        img_tensor = torch.clamp(img_tensor * factor, 0.0, 1.0)

    return img_tensor, mask_tensor


---

## Phase 2 - U-Net Architecture (Stronger V2)

The model is now a **4-level encoder-decoder U-Net** with skip connections, BatchNorm, dropout regularization, and He initialization for better convergence stability.

Each block uses:

$\text{Conv}(3\times3) \rightarrow \text{BN} \rightarrow \text{ReLU} \rightarrow \text{Conv}(3\times3) \rightarrow \text{BN} \rightarrow \text{ReLU}$

### Channel flow

| Stage | Channels |
|---|---|
| Encoder 1 | 1 -> 64 |
| Encoder 2 | 64 -> 128 |
| Encoder 3 | 128 -> 256 |
| Encoder 4 | 256 -> 512 |
| Bottleneck | 512 -> 1024 |
| Decoder 4 | 1024 -> 512 |
| Decoder 3 | 512 -> 256 |
| Decoder 2 | 256 -> 128 |
| Decoder 1 | 128 -> 64 |
| Output | 64 -> 1 (logits) |

The output layer returns **logits** (no sigmoid in the model head). Sigmoid is applied only when computing Dice/visualization/inference thresholding.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=64, dropout=0.2):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # Encoder (4 levels)
        self.enc1 = DoubleConv(in_channels, base_channels, dropout=0.0)
        self.enc2 = DoubleConv(base_channels, base_channels * 2, dropout=0.0)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4, dropout=0.1)
        self.enc4 = DoubleConv(base_channels * 4, base_channels * 8, dropout=0.1)

        # Bottleneck
        self.bottleneck = DoubleConv(base_channels * 8, base_channels * 16, dropout=dropout)

        # Decoder
        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base_channels * 16, base_channels * 8, dropout=0.1)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4, dropout=0.1)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2, dropout=0.0)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels, dropout=0.0)

        # Output logits (no sigmoid here)
        self.out = nn.Conv2d(base_channels, out_channels, kernel_size=1)

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.BatchNorm2d):
            nn.init.constant_(module.weight, 1)
            nn.init.constant_(module.bias, 0)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(device)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready on: {device} | trainable params: {num_params:,}")

### 2.1 Dice coefficient metric

Binary cross-entropy is used for optimization, but Dice is the key overlap metric for segmentation quality.

Dice is computed as:

$$
\text{Dice} = \frac{2|A \cap B| + \epsilon}{|A| + |B| + \epsilon}
$$

where $A$ is the thresholded prediction and $B$ is the ground-truth mask.

In [ ]:
# ── Dice coefficient that supports logits ─────────────────────────────────────
def dice_coeff(pred, target, threshold=0.5, eps=1e-6, from_logits=True):
    """Dice over batch. pred/target shape: (B,1,H,W)."""
    if from_logits:
        pred = torch.sigmoid(pred)
    pred_bin = (pred > threshold).float()

    intersection = (pred_bin * target).sum(dim=(1, 2, 3))
    denominator = pred_bin.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return dice.mean()

---

### 2.2 Training loop (AdamW + BCEWithLogits+Dice + early stopping)

| Hyperparameter | Value |
|---|---|
| Loss | $0.6\times\text{BCEWithLogits} + 0.4\times\text{DiceLoss}$ |
| Optimizer | AdamW |
| Learning rate | $3 \times 10^{-4}$ (adaptive via ReduceLROnPlateau) |
| Weight decay | $1 \times 10^{-4}$ |
| Batch size | 2 |
| Max epochs | 120 |
| Early stopping patience | 12 epochs without val-Dice improvement |
| Stability tricks | AMP (CUDA), gradient clipping (1.0) |

The loop logs train loss, train Dice, validation Dice, and learning rate.  
Best validation-Dice weights are saved immediately to preserve the strongest checkpoint.

In [ ]:
%pip install tqdm ipywidgets


In [ ]:
from tqdm.notebook import tqdm  # Progress bar for Jupyter / VS Code

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.6, dice_weight=0.4, eps=1e-6):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.eps = eps

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)

        intersection = (probs * targets).sum(dim=(1, 2, 3))
        denom = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + self.eps) / (denom + self.eps)
        dice_loss = 1.0 - dice.mean()

        total = self.bce_weight * bce_loss + self.dice_weight * dice_loss
        return total, bce_loss.detach(), dice_loss.detach()


criterion = BCEDiceLoss(bce_weight=0.6, dice_weight=0.4)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
 )

num_epochs = 120
patience = 12
best_dice = 0.0
epochs_no_imp = 0
best_ckpt = "breast_segmentation_unet_best.pth"

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

print(f"Starting Training on {device}...")
print(f"Early stopping: patience={patience}, monitoring=val Dice")
print("Loss: 0.6*BCEWithLogits + 0.4*DiceLoss")

epoch_train_losses = []
epoch_train_dices = []
epoch_val_dices = []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for imgs, masks in progress_bar:
        imgs, masks = augment(imgs, masks)
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(imgs)
            loss, bce_part, dice_part = criterion(logits, masks)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        current_loss = loss.item()
        current_dice = dice_coeff(logits.detach(), masks, from_logits=True).item()

        epoch_loss += current_loss
        epoch_dice += current_dice
        progress_bar.set_postfix(
            loss=f"{current_loss:.4f}",
            dice=f"{current_dice:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}"
        )

    avg_loss = epoch_loss / len(train_loader)
    avg_dice = epoch_dice / len(train_loader)

    model.eval()
    val_dice = 0.0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            val_dice += dice_coeff(logits, masks, from_logits=True).item()
    val_dice /= len(test_loader)

    scheduler.step(val_dice)

    epoch_train_losses.append(avg_loss)
    epoch_train_dices.append(avg_dice)
    epoch_val_dices.append(val_dice)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1:03d}/{num_epochs}] "
            f"Loss: {avg_loss:.4f}  Train Dice: {avg_dice:.4f}  "
            f"Val Dice: {val_dice:.4f}  LR: {optimizer.param_groups[0]['lr']:.1e}"
        )

    if val_dice > best_dice:
        best_dice = val_dice
        epochs_no_imp = 0
        torch.save(model.state_dict(), best_ckpt)
    else:
        epochs_no_imp += 1
        if epochs_no_imp >= patience:
            print(
                f"\nEarly stopping at epoch {epoch+1} — "
                f"no Val Dice improvement for {patience} epochs."
            )
            print(f"Best Val Dice: {best_dice:.4f}  →  weights saved to '{best_ckpt}'")
            break

torch.save(model.state_dict(), "breast_segmentation_unet.pth")
print("\nTraining complete.")
print("  Last weights : breast_segmentation_unet.pth")
print(f"  Best weights : {best_ckpt}  (Val Dice: {best_dice:.4f})")

In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

In [ ]:

pip install nvidia-pyindex

In [ ]:
pip install nvidia-cuda-runtime-cu12

In [ ]:
# Cell 17 (RTX 3060 optimized): Standalone GPU training
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Please switch to a CUDA-enabled environment.")

required = ["UNet", "full_dataset", "train_set", "test_set", "dice_coeff", "augment"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing prerequisites: {missing}. Run the setup cells for dataset and model utilities first."
    )

gpu_device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# RTX 3060-friendly defaults
train_batch_size = 4   # safe starting point for most RTX 3060 rigs
eval_batch_size = 4

train_loader_gpu = DataLoader(
    train_set,
    batch_size=train_batch_size,
    shuffle=True,
    pin_memory=True,
    num_workers=0,
 )
test_loader_gpu = DataLoader(
    test_set,
    batch_size=eval_batch_size,
    shuffle=False,
    pin_memory=True,
    num_workers=0,
 )

# Local loss class in case Cell 16 was not run
if "BCEDiceLoss" not in globals():
    class BCEDiceLoss(nn.Module):
        def __init__(self, bce_weight=0.6, dice_weight=0.4, eps=1e-6):
            super().__init__()
            self.bce = nn.BCEWithLogitsLoss()
            self.bce_weight = bce_weight
            self.dice_weight = dice_weight
            self.eps = eps

        def forward(self, logits, targets):
            bce_loss = self.bce(logits, targets)
            probs = torch.sigmoid(logits)
            inter = (probs * targets).sum(dim=(1, 2, 3))
            denom = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
            dice = (2.0 * inter + self.eps) / (denom + self.eps)
            dice_loss = 1.0 - dice.mean()
            total = self.bce_weight * bce_loss + self.dice_weight * dice_loss
            return total, bce_loss.detach(), dice_loss.detach()

# Fresh training objects (independent from Cell 16)
model = UNet().to(gpu_device, memory_format=torch.channels_last)
criterion = BCEDiceLoss(bce_weight=0.6, dice_weight=0.4)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
 )
scaler = torch.amp.GradScaler("cuda", enabled=True)

gpu_epochs = 60
gpu_patience = 10
epochs_no_imp = 0
best_gpu_dice = 0.0
best_gpu_ckpt = "breast_segmentation_unet_best_gpu.pth"

if "epoch_train_losses" not in globals():
    epoch_train_losses = []
if "epoch_train_dices" not in globals():
    epoch_train_dices = []
if "epoch_val_dices" not in globals():
    epoch_val_dices = []

print(f"Using GPU: {gpu_name}")
print(f"Starting standalone GPU training on: {gpu_device}")
print(f"Train batch size: {train_batch_size} | Eval batch size: {eval_batch_size}")
print(f"Epochs={gpu_epochs}, EarlyStop patience={gpu_patience}")

for ep in range(gpu_epochs):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0

    gpu_bar = tqdm(train_loader_gpu, desc=f"GPU Epoch {ep+1}/{gpu_epochs}", leave=False)
    for imgs, masks in gpu_bar:
        imgs, masks = augment(imgs, masks)
        imgs = imgs.to(gpu_device, non_blocking=True).contiguous(memory_format=torch.channels_last)
        masks = masks.to(gpu_device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=True):
            logits = model(imgs)
            loss, _, _ = criterion(logits, masks)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        batch_dice = dice_coeff(logits.detach(), masks, from_logits=True).item()
        epoch_loss += loss.item()
        epoch_dice += batch_dice
        gpu_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            dice=f"{batch_dice:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.1e}"
        )

    avg_loss = epoch_loss / len(train_loader_gpu)
    avg_dice = epoch_dice / len(train_loader_gpu)

    model.eval()
    val_dice = 0.0
    with torch.no_grad():
        for imgs, masks in test_loader_gpu:
            imgs = imgs.to(gpu_device, non_blocking=True).contiguous(memory_format=torch.channels_last)
            masks = masks.to(gpu_device, non_blocking=True)
            logits = model(imgs)
            val_dice += dice_coeff(logits, masks, from_logits=True).item()
    val_dice /= len(test_loader_gpu)

    scheduler.step(val_dice)

    epoch_train_losses.append(avg_loss)
    epoch_train_dices.append(avg_dice)
    epoch_val_dices.append(val_dice)

    if val_dice > best_gpu_dice:
        best_gpu_dice = val_dice
        epochs_no_imp = 0
        torch.save(model.state_dict(), best_gpu_ckpt)
    else:
        epochs_no_imp += 1
        if epochs_no_imp >= gpu_patience:
            print(
                f"\nEarly stopping at GPU epoch {ep+1} — "
                f"no Val Dice improvement for {gpu_patience} epochs."
            )
            break

    if (ep + 1) % 5 == 0 or ep == 0:
        print(
            f"[GPU {ep+1:03d}/{gpu_epochs}] Loss: {avg_loss:.4f} "
            f"Train Dice: {avg_dice:.4f}  Val Dice: {val_dice:.4f} "
            f"LR: {optimizer.param_groups[0]['lr']:.1e}"
        )

torch.save(model.state_dict(), "breast_segmentation_unet_gpu_last.pth")
print("\nStandalone GPU training complete.")
print(f"Best GPU Val Dice : {best_gpu_dice:.4f}")
print(f"Best GPU checkpoint: {best_gpu_ckpt}")
print("Last GPU checkpoint: breast_segmentation_unet_gpu_last.pth")

#### Optional manual checkpoint

Use this cell when you want an extra named snapshot (for example, after a specific epoch milestone).

This does not replace the best-checkpoint logic; it adds a manual archive point for experiments.

In [ ]:
torch.save(model.state_dict(), "breast_segmentation_unet_v1_60epochs.pth")
print("Weights saved successfully!")


### 2.3 Held-out test evaluation

This section evaluates the trained model on the held-out split (22% test subset).

Reported outputs:

- Mean BCE loss ± standard deviation
- Mean Dice score ± standard deviation

These values summarize both pixel-wise classification quality (BCE) and geometric overlap quality (Dice).

In [ ]:
# Evaluate on held-out test set (logits-aware)
model.eval()
test_losses = []
test_dices = []

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        loss, _, _ = criterion(logits, masks)

        test_losses.append(loss.item())
        test_dices.append(dice_coeff(logits, masks, from_logits=True).item())

print(f"Test Loss (BCE+Dice) : {np.mean(test_losses):.4f} ± {np.std(test_losses):.4f}")
print(f"Test Dice             : {np.mean(test_dices):.4f} ± {np.std(test_dices):.4f}")

---

### 2.4 Qualitative prediction check

This visualization plots one test sample as:

1. Input thermal image
2. Ground-truth mask
3. U-Net predicted mask

Use it as a quick sanity check before running full inference on all patient views.

In [ ]:
%pip install matplotlib


In [ ]:
import matplotlib.pyplot as plt

# 1. Visualize the Loss and Dice Scores
def plot_metrics(train_losses, val_dices, train_dices=None):
    epochs = range(1, len(train_losses) + 1)

    plt.figure(figsize=(12, 5))

    # Plot 1: BCE Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss (BCE+Dice)')
    plt.title('Training Loss over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot 2: Dice Coefficient (Validation vs Train if available)
    plt.subplot(1, 2, 2)
    if train_dices:
        plt.plot(epochs, train_dices, 'b-', label='Train Dice')
    plt.plot(epochs, val_dices, 'g-', label='Validation Dice', linewidth=2)
    
    # Highlight the best epoch
    best_epoch = val_dices.index(max(val_dices)) + 1
    plt.plot(best_epoch, max(val_dices), 'ro', label=f'Best Val Dice (Epoch {best_epoch})')

    plt.title('Dice Coefficient (Higher is Better)')
    plt.xlabel('Epochs')
    plt.ylabel('Dice Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

# Extract the tracked metrics from the training loop variables
# (Assuming you appended the avg_loss and val_dice to lists during the loop)
# Note: You will need to slightly modify the training loop above to append these values to lists:
# e.g., epoch_train_losses.append(avg_loss), epoch_val_dices.append(val_dice)

try:
    plot_metrics(epoch_train_losses, epoch_val_dices)
except NameError:
     print("Could not plot metrics. Ensure you are appending 'avg_loss' and 'val_dice' to lists named 'epoch_train_losses' and 'epoch_val_dices' inside your training loop.")

In [ ]:
# 2. Final Model Save Configuration (GPU checkpoint, no stale variable dependency)
print("\n--- Saving Final Model ---")
final_model_path = "breast_segmentation_unet_gpu.pth"
source_ckpt = "breast_segmentation_unet_best_gpu.pth"

if not os.path.exists(source_ckpt):
    raise FileNotFoundError(
        f"Required checkpoint not found: {source_ckpt}. "
        "Run GPU training first (Cell 20) to generate it."
    )

# Load the GPU-best weights and save under a clear final name
model.load_state_dict(torch.load(source_ckpt, weights_only=True))
torch.save(model.state_dict(), final_model_path)
print(f"Successfully saved the highest-performing model as: {final_model_path}")
print(f"Source checkpoint: {source_ckpt}")
print(f"You can now use '{final_model_path}' in Phase 3 for automated segmentation.")

In [ ]:
import matplotlib.pyplot as plt
import random

viz_ckpt = "breast_segmentation_unet_best_gpu.pth"
if not os.path.exists(viz_ckpt):
    raise FileNotFoundError(
        f"Visualization checkpoint not found: {viz_ckpt}. "
        "Run GPU training first (Cell 20)."
    )

# Always visualize with the newest GPU-best weights (avoid stale in-memory model state).
model.load_state_dict(torch.load(viz_ckpt, weights_only=True))
model.eval()

with torch.no_grad():
    sample_idx = random.randrange(len(test_set))
    img, mask = test_set[sample_idx]

    img = img.unsqueeze(0).to(device)
    mask = mask.unsqueeze(0)

    logits = model(img)
    pred = torch.sigmoid(logits)

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(img[0, 0].cpu(), cmap='magma')
    axs[0].set_title(f"Input Thermal (TIFF) - Sample #{sample_idx}")
    axs[1].imshow(mask[0, 0].cpu(), cmap='gray')
    axs[1].set_title("Manual Mask")
    axs[2].imshow(pred[0, 0].cpu(), cmap='gray')
    axs[2].set_title("U-Net Prediction")
    for ax in axs:
        ax.axis('off')
    plt.show()

---

## Inference Pipeline - Automated Segmentation and Edge Extraction

This phase applies the trained U-Net to all TIFF files and saves a clean edge map for each image.

Per image, the pipeline does:

1. Min-max normalization
2. Forward pass through U-Net
3. Threshold at 0.5 to create a binary mask
4. Keep only the largest connected contour (noise suppression)
5. Run Canny edge detection on the cleaned mask
6. Save output as <original_name>_edge.png while preserving folder structure

The edge outputs become the direct input for anchor extraction in Phase 3.

In [ ]:
output_edges = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\Automated_Edges3"


In [ ]:
def automated_segmentation(model_path, tiff_dir, output_dir):
    model = UNet().to(device)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()

    all_tiffs = glob.glob(os.path.join(tiff_dir, "**", "*.tiff"), recursive=True)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Running automated segmentation on {len(all_tiffs)} images...")
    print(f"Model checkpoint: {model_path}")
    print(f"Output edge dir : {output_dir}")

    for f_path in tqdm(all_tiffs):
        raw = tiff.imread(f_path)
        t_min, t_max = np.min(raw), np.max(raw)
        norm = (raw - t_min) / (t_max - t_min + 1e-8)
        img_256 = cv2.resize(norm.astype(np.float32), (256, 256))

        img_tensor = torch.from_numpy(img_256).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            pred_logits = model(img_tensor)
            pred = torch.sigmoid(pred_logits).squeeze().cpu().numpy()

        binary = (pred > 0.5).astype(np.uint8) * 255

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        clean_mask = np.zeros_like(binary)
        if not contours:
            print(f"Warning: no contour found for {os.path.basename(f_path)}, skipping.")
            continue

        largest = max(contours, key=cv2.contourArea)
        cv2.drawContours(clean_mask, [largest], -1, 255, thickness=cv2.FILLED)

        edges = cv2.Canny(clean_mask, 100, 200)

        rel_path = os.path.relpath(f_path, tiff_dir)
        out_name = os.path.splitext(rel_path)[0] + "_edge.png"
        save_path = os.path.join(output_dir, out_name)
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        cv2.imwrite(save_path, edges)


model_weights = "breast_segmentation_unet_best_gpu.pth"
output_edges = r"C:\Users\LENOVO THINKPAD T14\Documents\PROPOSAL TA\files\Rodriguez-Guerrero Dataset\Breast Thermography\3D Reconstruction\data\Automated_Edges3"

if not os.path.exists(model_weights):
    raise FileNotFoundError(f"Model checkpoint not found: {model_weights}")

automated_segmentation(model_weights, TIFF_BASE, output_edges)

---

### Edge-overlay quality check

This section randomly samples generated edge maps and overlays them in green on top of the original thermal image.

Why this is useful:

- Confirms that model boundaries align with anatomical breast contours.
- Quickly reveals failed masks, empty outputs, or corrupted files.
- Verifies that path handling remains stable on Windows/Unicode filenames.

The helper loader is intentionally robust against zero-byte and encoding-related read errors.

In [ ]:
def robust_read_image(path, flags=cv2.IMREAD_GRAYSCALE):
    """Bypasses Windows path encoding issues and handles empty files."""
    try:
        if os.path.getsize(path) == 0:  # Check if file was stopped mid-save
            return None
        # Read file as a numpy array buffer to bypass string encoding
        img_array = np.fromfile(path, dtype=np.uint8)
        img = cv2.imdecode(img_array, flags)
        return img
    except Exception:
        return None


def ultra_robust_light_test(tiff_dir, edge_dir, num_samples=3):
    # 1. Find all generated edge maps
    edge_files = glob.glob(os.path.join(edge_dir, "**", "*_edge.png"), recursive=True)

    if not edge_files:
        print("No edge files found! Check your Automated_Edges folder.")
        return

    # 2. Pick samples and setup plotting
    samples = random.sample(edge_files, min(num_samples, len(edge_files)))
    fig, axs = plt.subplots(len(samples), 2, figsize=(14, 6 * len(samples)))
    if len(samples) == 1:
        axs = np.expand_dims(axs, axis=0)

    for i, edge_path in enumerate(samples):
        # --- ROBUST READ ---
        edge_img = robust_read_image(edge_path)

        if edge_img is None:
            print(f"Skipping corrupted or unreadable edge map: {os.path.basename(edge_path)}")
            continue

        # 3. Find corresponding TIFF using Prefix Matching
        rel_path          = os.path.relpath(edge_path, edge_dir)
        patient_subfolder = os.path.dirname(rel_path)
        edge_filename     = os.path.basename(rel_path)
        search_folder     = os.path.join(tiff_dir, patient_subfolder)
        search_prefix     = edge_filename.split('(')[0].strip()

        potential_tiffs = glob.glob(os.path.join(search_folder, "*.tiff"))
        tiff_path = next((p for p in potential_tiffs if search_prefix in os.path.basename(p)), None)

        if tiff_path is None:
            continue

        # 4. Load/Normalize TIFF and Resize to 256x256
        raw      = tiff.imread(tiff_path)
        norm     = 255 * (raw - np.min(raw)) / (np.max(raw) - np.min(raw) + 1e-8)
        norm_256 = cv2.resize(norm.astype(np.uint8), (256, 256), interpolation=cv2.INTER_AREA)
        base_display = cv2.applyColorMap(norm_256, cv2.COLORMAP_MAGMA)

        # 5. Overlay edges
        overlay = base_display.copy()
        # The 'if edge_img is not None' check prevents the TypeError
        overlay[edge_img > 0] = [0, 255, 0]

        # Plotting
        axs[i, 0].imshow(cv2.cvtColor(base_display, cv2.COLOR_BGR2RGB))
        axs[i, 0].set_title(f"Thermal: {os.path.basename(tiff_path)}")
        axs[i, 1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axs[i, 1].set_title("Edge Overlay (Green)")
        for ax in axs[i]:
            ax.axis('off')

    plt.tight_layout()
    plt.show()


# Execute the test
ultra_robust_light_test(TIFF_BASE, output_edges, num_samples=3)


### Fine tuning with a smaller learning rate and more epochs

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HIGH-FIDELITY FINE-TUNING CELL FOR REGULAR UNET (newest) - TARGETING GPU 1
# ════════════════════════════════════════════════════════════════════════════
# Loads the best 89% checkpoint and applies a gentle learning rate of 2e-5
# with BCE + soft Dice Loss on CUDA Device 1 (GPU 1).

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path
from tqdm.notebook import tqdm

# 1. Configs
FT_CFG = {
    "lr": 2e-5,               # Gentle learning rate to avoid destroying learned features
    "epochs": 20,             # Short refinement period
    "best_unet_pth": "breast_segmentation_unet_best.pth",
    "save_pth": "breast_segmentation_unet_finetuned_best.pth"
}

# 2. Define Loss & Metric Functions
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, eps=1e-6):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.eps = eps

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(1, 2, 3))
        denom = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice = (2.0 * intersection + self.eps) / (denom + self.eps)
        dice_loss = 1.0 - dice.mean()
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

def dice_coeff(pred, target, threshold=0.5, eps=1e-6, from_logits=True):
    if from_logits:
        pred = torch.sigmoid(pred)
    pred_bin = (pred > threshold).float()
    intersection = (pred_bin * target).sum(dim=(1, 2, 3))
    denominator = pred_bin.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return ((2.0 * intersection + eps) / (denominator + eps)).mean()

# 3. Execution function targeting GPU 1
def fine_tune_segmentor(cfg):
    # Target GPU 1 explicitly if available
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        device = torch.device('cuda:1')
    elif torch.cuda.is_available():
        device = torch.device('cuda:0')
        print("WARNING: Only 1 GPU found. Defaulting to GPU 0.")
    else:
        device = torch.device('cpu')
        
    print(f"Fine-tuning on device: {device}")
    
    # Initialize same architecture
    model_ft = UNet().to(device)
    
    # Load 89% checkpoint
    ckpt_path = Path(cfg["best_unet_pth"])
    if ckpt_path.exists():
        print(f" Loading best segmentor weights from: {ckpt_path}")
        model_ft.load_state_dict(torch.load(ckpt_path, map_location=device))
        print(" Model loaded successfully! Ready for refinement.")
    else:
        raise FileNotFoundError(f"Base checkpoint not found at {ckpt_path}. Train the base model first!")

    criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
    optimizer = optim.AdamW(model_ft.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    
    # Evaluate initial performance on test loader
    model_ft.eval()
    init_dice = 0.0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model_ft(imgs)
            init_dice += dice_coeff(logits, masks, from_logits=True).item()
    init_dice /= len(test_loader)
    print(f"Initial Test Dice Score: {init_dice:.4f}")

    best_dice = init_dice
    
    # Fine-Tuning Loop
    for epoch in range(cfg["epochs"]):
        model_ft.train()
        epoch_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f"FT Epoch {epoch+1}/{cfg['epochs']}", leave=False)
        for imgs, masks in progress_bar:
            # Apply conservative augmentations (matching original pipeline)
            imgs, masks = augment(imgs, masks)
            imgs, masks = imgs.to(device), masks.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            logits = model_ft(imgs)
            loss = criterion(logits, masks)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model_ft.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")
            
        # Validation
        model_ft.eval()
        val_dice = 0.0
        with torch.no_grad():
            for imgs, masks in test_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = model_ft(imgs)
                val_dice += dice_coeff(logits, masks, from_logits=True).item()
        val_dice /= len(test_loader)
        
        print(f"Epoch [{epoch+1:02d}/{cfg['epochs']}] | Loss: {epoch_loss/len(train_loader):.4f} | Test Dice: {val_dice:.4f}")
        
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model_ft.state_dict(), cfg["save_pth"])
            print(f"  ★ New best segmentor dice={best_dice:.4f}! Saved to '{cfg['save_pth']}'")
            
    print(f"\n Fine-Tuning complete. Best Segmentor Test Dice: {best_dice:.4f}")

# Run fine-tuning
fine_tune_segmentor(FT_CFG)


In [ ]:
import os
import glob
import cv2
import numpy as np
import tifffile as tiff
import torch
from tqdm.notebook import tqdm

# --- Configuration ---
DATA_DIR = r"..\..\data\organized_by_patient"
OUTPUT_UNET_DIR = r"..\..\data\organized_by_patient_unet"
os.makedirs(OUTPUT_UNET_DIR, exist_ok=True)

# Path to the best weights (make sure this file exists!)
# You might need to change this path if your weights are saved somewhere else.
WEIGHTS_PATH = "breast_segmentation_unet_best.pth" 

def load_thermal_for_unet(path):
    """Load and normalize thermal TIFF to float32 [0, 1]."""
    raw = tiff.imread(path).astype(np.float32)
    mn, mx = raw.min(), raw.max()
    if mx - mn < 1e-6:
        normalized = np.zeros_like(raw)
    else:
        normalized = (raw - mn) / (mx - mn)
    
    # Resize to 256x256 for U-Net input
    img_256 = cv2.resize(normalized, (256, 256), interpolation=cv2.INTER_AREA)
    
    # Convert to tensor (1, 1, 256, 256)
    tensor = torch.from_numpy(img_256).unsqueeze(0).unsqueeze(0)
    return tensor

def refine_unet_mask(mask_prob, threshold=0.5):
    """
    Costa et al. Morphological Refinement Step
    Removes floating blobs by keeping only the largest connected breast shape.
    """
    binary = (mask_prob > threshold).astype(np.uint8)
    
    # Find distinct blobs
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        return np.zeros_like(binary)
        
    # Find the largest blob (the breast) and delete everything else
    largest_contour = max(contours, key=cv2.contourArea)
    refined_mask = np.zeros_like(binary)
    cv2.drawContours(refined_mask, [largest_contour], -1, 255, thickness=cv2.FILLED)
    
    return refined_mask

def generate_unet_masks():
    if not os.path.exists(WEIGHTS_PATH):
        print(f"ERROR: Could not find weights at {WEIGHTS_PATH}")
        print("Please train the model or update the path.")
        return
        
    # Load weights into the model
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    model.eval()
    
    patient_folders = [f for f in os.listdir(DATA_DIR) if f.startswith('Patient_')]
    print(f"Found {len(patient_folders)} patients. Starting U-Net extraction...")
    
    processed_count = 0
    
    with torch.no_grad():
        for patient in tqdm(patient_folders, desc="Processing U-Net Masks"):
            patient_path = os.path.join(DATA_DIR, patient)
            
            subdirs = [d for d in os.listdir(patient_path) if os.path.isdir(os.path.join(patient_path, d))]
            category_name = subdirs[0] if subdirs else ""
            input_subfolder = os.path.join(patient_path, category_name)
            
            out_subfolder = os.path.join(OUTPUT_UNET_DIR, patient, category_name)
            os.makedirs(out_subfolder, exist_ok=True)
            
            tiff_files = glob.glob(os.path.join(input_subfolder, "*.tiff"))
            
            for tiff_path in tiff_files:
                view_name = os.path.basename(tiff_path).replace('.tiff', '')
                
                # 1. Load and prep for U-Net
                tensor = load_thermal_for_unet(tiff_path)
                tensor = tensor.to(device)
                
                # 2. Predict
                logits = model(tensor)
                probs = torch.sigmoid(logits).squeeze().cpu().numpy()
                
                # 3. Refine (remove floating blobs)
                refined_mask_256 = refine_unet_mask(probs, threshold=0.5)
                
                # 4. Resize down to 128x128 so it matches Otsu masks perfectly for 3D-BreastNet
                final_mask_128 = cv2.resize(refined_mask_256, (128, 128), interpolation=cv2.INTER_NEAREST)
                
                # 5. Save safely bypassing Windows encoding bug for degree (°) symbols
                out_file = os.path.join(out_subfolder, f"{view_name}_unet_mask.png")
                _, buf = cv2.imencode('.png', final_mask_128)
                buf.tofile(out_file)
                
                processed_count += 1

    print(f"Done! Generated {processed_count} U-Net masks.")

# Run it!
generate_unet_masks()
